# Calibration and Fusion Project

## 1. Project goal

The goal of this part of the project is to take the best-performing classifiers from the previous laboratories, calibrate their scores using a target application prior, evaluate the calibrated systems with Bayes decision metrics, and then study score-level fusion. The emphasis is not just on classification accuracy, but on how the raw scores behave as log-likelihood-ratio-like quantities under different operating points.

In practice, this means three things. First, we reuse the best classifier of each family selected earlier in the project. Second, we train a calibration transformation using the validation scores and a K-fold strategy. Third, we apply the final calibrated models to the held-out evaluation set and interpret the resulting actual and minimum DCF values together with the Bayes error plots.

In [ ]:

import numpy as np
import scipy.optimize
import scipy.special
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 140

def vcol(x):
    return x.reshape((x.size, 1))

def vrow(x):
    return x.reshape((1, x.size))


## 2. Data loading and split

The calibration part of the assignment uses the validation scores produced by the earlier project notebooks. The evaluation set must remain untouched until the very end, so that the final result is unbiased with respect to model selection. The data format is the standard one used throughout the course: features in the first columns and the class label in the last column.

In [ ]:

def load_data(fname):
    D, L = [], []
    with open(fname) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 2:
                continue
            D.append(list(map(float, parts[:-1])))
            L.append(int(parts[-1]))
    return np.array(D).T, np.array(L)

DTR, LTR = load_data('../trainData.txt')
DEVAL, LEVAL = load_data('../evalData.txt')

def split_db_2to1(D, L, seed=0):
    nTrain = int(D.shape[1] * 2.0 / 3.0)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    return (D[:, idx[:nTrain]], L[idx[:nTrain]]), (D[:, idx[nTrain:]], L[idx[nTrain:]])

(DTR_tr, LTR_tr), (DTR_val, LTR_val) = split_db_2to1(DTR, LTR)
pT = 0.2

print('Training split:', DTR_tr.shape)
print('Validation split:', DTR_val.shape)
print('Evaluation set:', DEVAL.shape)


## 3. Bayes-risk utilities

The project uses minimum DCF and actual DCF because these are the metrics that matter for decision-making under a specific prior. The minimum DCF tells us how good the score ranking is if we were free to move the threshold optimally. The actual DCF tells us how well the chosen score scale matches the Bayes threshold of the target application.

These functions are the same ones used in the course notebooks, with a fast sweep over all possible thresholds for the minimum DCF computation.

In [ ]:

def compute_confusion_matrix(pred, labels):
    nC = labels.max() + 1
    M = np.zeros((nC, nC), dtype=np.int32)
    for i in range(labels.size):
        M[pred[i], labels[i]] += 1
    return M

def compute_empirical_Bayes_risk_binary(pred, labels, prior, Cfn, Cfp, normalize=True):
    M = compute_confusion_matrix(pred, labels)
    Pfn = M[0,1] / (M[0,1] + M[1,1])
    Pfp = M[1,0] / (M[0,0] + M[1,0])
    be = prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp
    if normalize:
        return be / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    return be

def compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp):
    threshold = -np.log((prior * Cfn) / ((1 - prior) * Cfp))
    return np.int32(llr > threshold)

def compute_actDCF_binary_fast(llr, labels, prior, Cfn, Cfp, normalize=True):
    pred = compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp)
    return compute_empirical_Bayes_risk_binary(pred, labels, prior, Cfn, Cfp, normalize)

def compute_Pfn_Pfp_allThresholds_fast(llr, labels):
    sorter = np.argsort(llr)
    llrS = llr[sorter]
    labS = labels[sorter]
    nT = (labS == 1).sum()
    nF = (labS == 0).sum()
    nFN = 0
    nFP = nF
    Pfn = [nFN / nT]
    Pfp = [nFP / nF]
    for idx in range(len(llrS)):
        if labS[idx] == 1:
            nFN += 1
        else:
            nFP -= 1
        Pfn.append(nFN / nT)
        Pfp.append(nFP / nF)
    llrS2 = np.concatenate([[-np.inf], llrS])
    PfnO, PfpO, thO = [], [], []
    for idx in range(len(llrS2)):
        if idx == len(llrS2) - 1 or llrS2[idx + 1] != llrS2[idx]:
            PfnO.append(Pfn[idx])
            PfpO.append(Pfp[idx])
            thO.append(llrS2[idx])
    return np.array(PfnO), np.array(PfpO), np.array(thO)

def compute_minDCF_binary_fast(llr, labels, prior, Cfn, Cfp):
    Pfn, Pfp, _ = compute_Pfn_Pfp_allThresholds_fast(llr, labels)
    dcf = (prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp) / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    return dcf.min()

def bayesPlot(S, L, left=-3, right=3, npts=21):
    elos = np.linspace(left, right, npts)
    eps = 1.0 / (1.0 + np.exp(-elos))
    act = [compute_actDCF_binary_fast(S, L, e, 1, 1) for e in eps]
    mn = [compute_minDCF_binary_fast(S, L, e, 1, 1) for e in eps]
    return elos, act, mn


## 4. Model helpers

Calibration and fusion are both implemented with weighted logistic regression. The only difference is the dimensionality of the input: one score for calibration of a single system, multiple scores for fusion. The final affine output is then corrected by subtracting the prior log-odds so that the result can be interpreted as a log-likelihood ratio.

The notebook also includes the model code needed to generate the best scores for the three families selected in the previous project parts: logistic regression, SVM, and GMM.

In [ ]:

def trainWeightedLogRegBinary(DTR, LTR, l, pT):
    ZTR = LTR * 2.0 - 1.0
    wT = pT / (ZTR > 0).sum()
    wN = (1 - pT) / (ZTR < 0).sum()
    def obj(v):
        w = v[:-1]
        b = v[-1]
        s = np.dot(vcol(w).T, DTR).ravel() + b
        loss = np.logaddexp(0, -ZTR * s)
        loss[ZTR > 0] *= wT
        loss[ZTR < 0] *= wN
        G = -ZTR / (1 + np.exp(ZTR * s))
        G[ZTR > 0] *= wT
        G[ZTR < 0] *= wN
        GW = (vrow(G) * DTR).sum(1) + l * w.ravel()
        Gb = G.sum()
        return loss.sum() + l / 2 * np.linalg.norm(w)**2, np.hstack([GW, [Gb]])
    vf = scipy.optimize.fmin_l_bfgs_b(obj, x0=np.zeros(DTR.shape[0] + 1), iprint=-1)[0]
    return vf[:-1], vf[-1]

def apply_cal(w, b, s, pT):
    return (w.T @ vrow(s) + b - np.log(pT / (1 - pT))).ravel()

def trainSVM_linear(DTR, LTR, C, pT=None):
    ZTR = (2 * LTR - 1).astype(float)
    n = DTR.shape[1]
    Ci = np.where(LTR == 1, C * pT / ((LTR == 1).sum()) * n,
                  C * (1 - pT) / ((LTR == 0).sum()) * n) if pT else np.full(n, C)
    DTRex = np.vstack([DTR, np.ones((1, n))])
    ZDTRex = vrow(ZTR) * DTRex
    H = ZDTRex.T @ ZDTRex
    def dual_obj(a):
        Ha = H @ a
        return 0.5 * (a @ Ha) - a.sum(), Ha - np.ones(n)
    a, _, _ = scipy.optimize.fmin_l_bfgs_b(dual_obj, np.zeros(n), bounds=[(0, ci) for ci in Ci], iprint=-1)
    wh = ZDTRex @ a
    return wh[:-1], wh[-1]

def logpdf_GAU_ND(X, mu, C):
    M = X.shape[0]
    _, ld = np.linalg.slogdet(C)
    diff = X - mu
    inv = np.linalg.inv(C)
    return -0.5 * (M * np.log(2*np.pi) + ld + (diff * (inv @ diff)).sum(0))

def GMM_EM(X, gmm, niter=100, psi=0.01):
    for _ in range(niter):
        logS = np.array([w + logpdf_GAU_ND(X, mu, C) for (w, mu, C) in gmm])
        gamma = np.exp(logS - scipy.special.logsumexp(logS, axis=0))
        new_gmm = []
        for g in range(len(gmm)):
            Z = gamma[g].sum()
            F = (gamma[g:g+1] * X).sum(1, keepdims=True)
            S = (gamma[g:g+1] * X) @ X.T
            mu_new = F / Z
            C_new = S / Z - mu_new @ mu_new.T
            U, s, _ = np.linalg.svd(C_new)
            s = np.maximum(s, psi)
            new_gmm.append((np.log(Z / X.shape[1]), mu_new, U @ np.diag(s) @ U.T))
        gmm = new_gmm
    return gmm

def GMM_train(X, nComp, psi=0.01):
    mu0 = X.mean(1, keepdims=True)
    Dc = X - mu0
    C0 = Dc @ Dc.T / X.shape[1]
    U, s, _ = np.linalg.svd(C0)
    s = np.maximum(s, psi)
    C0 = U @ np.diag(s) @ U.T
    gmm = [(0.0, mu0, C0)]
    while len(gmm) < nComp:
        ng = []
        for (w, mu, C) in gmm:
            U, s, _ = np.linalg.svd(C)
            d = U[:, 0:1] * s[0]**0.5 * 0.1
            ng.append((w - np.log(2), mu + d, C))
            ng.append((w - np.log(2), mu - d, C))
        gmm = GMM_EM(X, ng, psi=psi)
    return gmm

def GMM_ll(X, gmm):
    logS = np.array([w + logpdf_GAU_ND(X, mu, C) for (w, mu, C) in gmm])
    return scipy.special.logsumexp(logS, axis=0)


## 5. Best models from the previous project parts

The best-performing classifier of each family is chosen according to the earlier notebooks. In the polished version, we keep those choices explicit so the notebook is easy to read and easy to defend during a presentation or oral exam.

In [ ]:

# Best model choices from the previous project parts
best_lr_lambda = 1e-2
best_svm_C = 1e-1
best_gmm_nComp = 4

w_lr_base, b_lr_base = trainWeightedLogRegBinary(DTR_tr, LTR_tr, best_lr_lambda, pT)
w_svm_base, b_svm_base = trainSVM_linear(DTR_tr, LTR_tr, best_svm_C, pT=pT)
best_g0 = GMM_train(DTR_tr[:, LTR_tr == 0], best_gmm_nComp)
best_g1 = GMM_train(DTR_tr[:, LTR_tr == 1], best_gmm_nComp)

lr_val_raw = w_lr_base @ DTR_val + b_lr_base - np.log(pT / (1 - pT))
svm_val_raw = w_svm_base @ DTR_val + b_svm_base
gmm_val_raw = GMM_ll(DTR_val, best_g1) - GMM_ll(DTR_val, best_g0)

print('Validation raw scores computed.')
print('LR  minDCF=', round(compute_minDCF_binary_fast(lr_val_raw, LTR_val, pT, 1, 1), 4), 'actDCF=', round(compute_actDCF_binary_fast(lr_val_raw, LTR_val, pT, 1, 1), 4))
print('SVM minDCF=', round(compute_minDCF_binary_fast(svm_val_raw, LTR_val, pT, 1, 1), 4), 'actDCF=', round(compute_actDCF_binary_fast(svm_val_raw, LTR_val, pT, 1, 1), 4))
print('GMM minDCF=', round(compute_minDCF_binary_fast(gmm_val_raw, LTR_val, pT, 1, 1), 4), 'actDCF=', round(compute_actDCF_binary_fast(gmm_val_raw, LTR_val, pT, 1, 1), 4))


## 6. K-fold calibration

For each system we perform five-fold calibration on the validation scores. This is the recommended strategy because it uses the limited validation data more efficiently and avoids overfitting the calibration parameters to a single split.

The calibration model is always trained with the target prior, here \#(\pi_T = 0.2\). The affine transform is fitted on the training folds and applied to the held-out fold. The pooled calibrated scores are then evaluated with both minimum DCF and actual DCF.

In [ ]:

def kfold_calibrate(scores, labels, pT, K=5):
    cal_scores = []
    cal_labels = []
    for idx in range(K):
        SCAL = np.hstack([scores[jdx::K] for jdx in range(K) if jdx != idx])
        SVAL = scores[idx::K]
        LCAL = np.hstack([labels[jdx::K] for jdx in range(K) if jdx != idx])
        LVAL = labels[idx::K]
        w, b = trainWeightedLogRegBinary(vrow(SCAL), LCAL, 0, pT)
        cal = (w.T @ vrow(SVAL) + b - np.log(pT / (1 - pT))).ravel()
        cal_scores.append(cal)
        cal_labels.append(LVAL)
    return np.hstack(cal_scores), np.hstack(cal_labels)

val_scores = {'LR': lr_val_raw, 'SVM': svm_val_raw, 'GMM': gmm_val_raw}
kfold_results = {}
final_models = {}

for name, s in val_scores.items():
    cs, cl = kfold_calibrate(s, LTR_val, pT)
    kfold_results[name] = {
        'cal_scores': cs,
        'cal_labels': cl,
        'minDCF_raw': compute_minDCF_binary_fast(s, LTR_val, pT, 1, 1),
        'actDCF_raw': compute_actDCF_binary_fast(s, LTR_val, pT, 1, 1),
        'minDCF_cal': compute_minDCF_binary_fast(cs, cl, pT, 1, 1),
        'actDCF_cal': compute_actDCF_binary_fast(cs, cl, pT, 1, 1),
    }
    final_models[name] = trainWeightedLogRegBinary(vrow(s), LTR_val, 0, pT)

for name, r in kfold_results.items():
    print(f"{name}: raw minDCF={r['minDCF_raw']:.4f}, raw actDCF={r['actDCF_raw']:.4f}, cal minDCF={r['minDCF_cal']:.4f}, cal actDCF={r['actDCF_cal']:.4f}")


## 7. Score-level fusion

Fusion is obtained by stacking the system scores and training the same weighted logistic-regression calibration model on the score matrix. The resulting affine function is now a fusion rule because it combines several systems at once instead of transforming one score alone.

The K-fold fusion setup follows the same philosophy as the single-system calibration: each fold is held out once, fused using the model trained on the remaining folds, and then pooled to obtain the final performance estimate.

In [ ]:

all_val_matrix = np.vstack([val_scores['LR'], val_scores['SVM'], val_scores['GMM']])
fused_cal = []
fused_lab = []
for idx in range(5):
    SCAL = np.hstack([all_val_matrix[:, jdx::5] for jdx in range(5) if jdx != idx])
    SVAL = all_val_matrix[:, idx::5]
    LCAL = np.hstack([LTR_val[jdx::5] for jdx in range(5) if jdx != idx])
    LVAL = LTR_val[idx::5]
    w, b = trainWeightedLogRegBinary(SCAL, LCAL, 0, pT)
    fused_cal.append((w.T @ SVAL + b - np.log(pT / (1 - pT))).ravel())
    fused_lab.append(LVAL)

fused_cal_scores = np.hstack(fused_cal)
fused_cal_labels = np.hstack(fused_lab)
w_fus, b_fus = trainWeightedLogRegBinary(all_val_matrix, LTR_val, 0, pT)

print('Fusion validation minDCF=', round(compute_minDCF_binary_fast(fused_cal_scores, fused_cal_labels, pT, 1, 1), 4))
print('Fusion validation actDCF=', round(compute_actDCF_binary_fast(fused_cal_scores, fused_cal_labels, pT, 1, 1), 4))


## 8. Evaluation set

This is the final evaluation stage. At this point, no more model fitting is allowed on the evaluation data. We simply apply the already trained calibration and fusion models and measure the resulting performance. This is the only unbiased way to report the final behavior of the delivered system.

In [ ]:

w_lr_f, b_lr_f = trainWeightedLogRegBinary(DTR_tr, LTR_tr, best_lr_lambda, pT)
eval_lr_raw = w_lr_f @ DEVAL + b_lr_f - np.log(pT / (1 - pT))

w_svm_f, b_svm_f = trainSVM_linear(DTR_tr, LTR_tr, best_svm_C, pT=pT)
eval_svm_raw = w_svm_f @ DEVAL + b_svm_f

eval_gmm_raw = GMM_ll(DEVAL, best_g1) - GMM_ll(DEVAL, best_g0)

lr_w_cal, lr_b_cal = final_models['LR']
svm_w_cal, svm_b_cal = final_models['SVM']
gmm_w_cal, gmm_b_cal = final_models['GMM']

eval_lr_cal = apply_cal(lr_w_cal, lr_b_cal, eval_lr_raw, pT)
eval_svm_cal = apply_cal(svm_w_cal, svm_b_cal, eval_svm_raw, pT)
eval_gmm_cal = apply_cal(gmm_w_cal, gmm_b_cal, eval_gmm_raw, pT)
eval_fused = (w_fus.T @ np.vstack([eval_lr_raw, eval_svm_raw, eval_gmm_raw]) + b_fus - np.log(pT / (1 - pT))).ravel()

print('Evaluation results at pT=0.2')
for name, s in [('LR raw', eval_lr_raw), ('LR cal', eval_lr_cal), ('SVM raw', eval_svm_raw), ('SVM cal', eval_svm_cal), ('GMM raw', eval_gmm_raw), ('GMM cal', eval_gmm_cal), ('Fusion', eval_fused)]:
    print(f"{name:10s}  minDCF={compute_minDCF_binary_fast(s, LEVAL, pT, 1, 1):.4f}  actDCF={compute_actDCF_binary_fast(s, LEVAL, pT, 1, 1):.4f}")


## 9. Bayes error plots

The Bayes error plots are the most informative way to check calibration. A well-calibrated system should keep its actual DCF close to the minimum DCF over a broad range of prior log-odds, not only at the target prior.

The plots below compare the raw and calibrated systems on both the validation and evaluation datasets, and they also show the fused model.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Calibration & Fusion — Bayes Error Plots', fontsize=14)
colors = {'LR': 'C0', 'SVM': 'C1', 'GMM': 'C2', 'Fusion': 'C3'}

ax = axes[0, 0]
for name in ['LR', 'SVM', 'GMM']:
    r = kfold_results[name]
    lo, act, mn = bayesPlot(r['cal_scores'], r['cal_labels'])
    lo2, act_r, _ = bayesPlot(val_scores[name], LTR_val)
    ax.plot(lo, mn, color=colors[name], ls='--', label=f'{name} minDCF')
    ax.plot(lo2, act_r, color=colors[name], ls=':', alpha=0.5)
    ax.plot(lo, act, color=colors[name], ls='-', label=f'{name} actDCF (cal)')
ax.set_ylim(0, 0.8)
ax.set_title('Validation: individual systems')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')
ax.set_ylabel('normalized DCF')

ax = axes[0, 1]
for name in ['LR', 'SVM', 'GMM']:
    r = kfold_results[name]
    lo, act, mn = bayesPlot(r['cal_scores'], r['cal_labels'])
    ax.plot(lo, mn, color=colors[name], ls='--', label=f'{name} minDCF')
    ax.plot(lo, act, color=colors[name], ls='-', label=f'{name} actDCF')
lo, af, mf = bayesPlot(fused_cal_scores, fused_cal_labels)
ax.plot(lo, mf, 'C3--', label='Fusion minDCF')
ax.plot(lo, af, 'C3-', label='Fusion actDCF')
ax.set_ylim(0, 0.8)
ax.set_title('Validation: fusion')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')

ax = axes[0, 2]
lo, ac, mn = bayesPlot(kfold_results['GMM']['cal_scores'], kfold_results['GMM']['cal_labels'])
lo2, ar, _ = bayesPlot(val_scores['GMM'], LTR_val)
ax.plot(lo, mn, 'C2--', label='minDCF')
ax.plot(lo2, ar, 'C2:', label='actDCF (pre-cal)')
ax.plot(lo, ac, 'C2-', label='actDCF (cal)')
ax.set_ylim(0, 0.5)
ax.set_title('Validation: GMM')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')

ax = axes[1, 0]
for name, sr, sc in [('LR', eval_lr_raw, eval_lr_cal), ('SVM', eval_svm_raw, eval_svm_cal), ('GMM', eval_gmm_raw, eval_gmm_cal)]:
    lo, ar, mn = bayesPlot(sr, LEVAL)
    lo2, ac, _ = bayesPlot(sc, LEVAL)
    ax.plot(lo, mn, color=colors[name], ls='--', label=f'{name} minDCF')
    ax.plot(lo, ar, color=colors[name], ls=':', alpha=0.5)
    ax.plot(lo2, ac, color=colors[name], ls='-', label=f'{name} actDCF (cal)')
ax.set_ylim(0, 0.8)
ax.set_title('Evaluation: individual systems')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')
ax.set_ylabel('normalized DCF')

ax = axes[1, 1]
lo, af, mf = bayesPlot(eval_fused, LEVAL)
for name, sc in [('LR', eval_lr_cal), ('SVM', eval_svm_cal), ('GMM', eval_gmm_cal)]:
    lo2, ac, mn = bayesPlot(sc, LEVAL)
    ax.plot(lo2, mn, color=colors[name], ls='--', label=f'{name} minDCF')
    ax.plot(lo2, ac, color=colors[name], ls='-', label=f'{name} actDCF')
ax.plot(lo, mf, 'C3--', label='Fusion minDCF')
ax.plot(lo, af, 'C3-', label='Fusion actDCF')
ax.set_ylim(0, 0.8)
ax.set_title('Evaluation: fusion')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')

ax = axes[1, 2]
lo, ac, mn = bayesPlot(eval_gmm_cal, LEVAL)
lo2, ar, _ = bayesPlot(eval_gmm_raw, LEVAL)
ax.plot(lo, mn, 'C2--', label='minDCF')
ax.plot(lo2, ar, 'C2:', label='actDCF (pre-cal)')
ax.plot(lo, ac, 'C2-', label='actDCF (cal)')
ax.set_ylim(0, 0.5)
ax.set_title('Evaluation: GMM')
ax.legend(fontsize=8)
ax.set_xlabel('prior log-odds')

plt.tight_layout()
plt.savefig('project_bayes_plots_polished.png', dpi=180, bbox_inches='tight')
plt.show()


## 10. Answers to the project questions

### What happens after calibration?
Calibration is most useful when the raw scores are not already LLR-like. That is why the SVM benefits the most: its actDCF is much larger than its minDCF before calibration, which means the ranking is acceptable but the score scale is poorly matched to the Bayes threshold. After calibration, the actDCF becomes much closer to the minDCF, especially around the target prior.

The logistic-regression system changes less because it already produces scores that behave more like log-likelihood ratios. In other words, there is less to fix. The GMM is also already well calibrated, so the gain from calibration is small or negligible. This is exactly what we expect from a generative classifier that is already directly modeling class-conditional densities.

### What do the Bayes error plots show?
The Bayes error plots are important because they tell us whether the improvement is only local at the target prior or more general across different applications. The SVM calibration curve becomes much closer to the minimum-DCF curve over a large region of prior log-odds, so the calibration is not just tuned to one point. The GMM stays close to its own minimum-DCF curve even before calibration, which confirms that it already had a score scale close to the ideal one.

### Is fusion useful?
Fusion helps when the systems provide complementary information. In this project, fusion improves over the weaker systems, but it does not clearly beat the best GMM on the evaluation set. That means the systems are not diverse enough to gain a large boost from linear combination, or the strongest system already captures most of the discriminative structure. The fusion is still reasonable, but it is not the best delivered system for this dataset.

### Which final system should be delivered?
The delivered system should be the **calibrated GMM**. The decision should be based on actual DCF, not minimum DCF, because calibration has already been addressed and the final application must use the Bayes threshold directly. The calibrated GMM gives the best balance between discrimination and calibration on the evaluation set. It also behaves consistently across the Bayes error plots, which makes it a safer deployment choice than the fused model in this particular experiment.

### General conclusion
The most important lesson of this project is that good ranking is not enough. A model can have a good minimum DCF and still have a bad actual DCF if its scores are not calibrated. The calibration stage fixes that mismatch, and the K-fold version is the most robust because it uses more of the available validation data while still avoiding a single fragile split. The final model should therefore be judged by actual DCF together with the Bayes error plots, not by minimum DCF alone.